In [ ]:
# ============================================================
# E02: Synthetic / Semi-Synthetic Uplift Dataset Generator
# ============================================================
#
# Supports:
#   - synthetic-01
#   - synthetic-02
#   - synthetic-hillstrom
#
# Goal:
# Create reusable synthetic datasets for:
#   - conversion ranking
#   - uplift modeling
#   - policy allocation
#   - regret benchmarking
#   - feedback loop simulations
#
# Output:
#   pandas DataFrame with:
#
#   X features
#   treatment assignment
#   potential outcomes
#   observed outcomes
#   uplift
#   revenue
#   intervention cost
#
# ============================================================

from dataclasses import dataclass
from typing import Optional
import numpy as np
import pandas as pd

from scipy.special import expit


# ============================================================
# CONFIG
# ============================================================

@dataclass
class SyntheticConfig:
    dataset_type: str = "synthetic-01"

    n_samples: int = 100_000
    random_state: int = 42

    treatment_rate: float = 0.5
    noise_std: float = 0.0
    observational: bool = False
    include_revenue: bool = True
    clip_tau: bool = True


# ============================================================
# UTILITIES
# ============================================================

def sigmoid(x):
    return expit(x)


def sample_binary(prob, rng):
    return rng.binomial(1, prob)


# ============================================================
# FEATURE GENERATION
# ============================================================

def generate_base_features(n, rng):

    segment = rng.choice(
        ["high_intent", "mid_intent", "low_intent", "enterprise", "bad_fit"],
        size=n,
        p=[0.20, 0.35, 0.25, 0.10, 0.10]
    )

    source = rng.choice(
        ["organic", "paid", "referral", "direct"],
        size=n,
        p=[0.35, 0.35, 0.15, 0.15]
    )

    device = rng.choice(
        ["Android", "iOS", "desktop", "tablet"],
        size=n,
        p=[0.40, 0.27, 0.35, 0.02]
    )

    engagement_score = rng.beta(2, 5, size=n)

    pages_viewed = rng.poisson(4, size=n)

    form_started = rng.binomial(1, 0.40, size=n)

    otp_completed = rng.binomial(1, 0.18, size=n)

    days_since_last_visit = rng.gamma(2, 3, size=n)

    company_size = rng.choice(
        ["small", "mid", "large"],
        size=n,
        p=[0.60, 0.30, 0.10]
    )

    lead_value = np.clip(
        rng.normal(1200, 400, size=n),
        100,
        None
    )

    intervention_cost = np.clip(
        rng.normal(40, 15, size=n),
        5,
        None
    )

    return pd.DataFrame({
        "segment": segment,
        "source": source,
        "device": device,
        "engagement_score": engagement_score,
        "pages_viewed": pages_viewed,
        "form_started": form_started,
        "otp_completed": otp_completed,
        "days_since_last_visit": days_since_last_visit,
        "company_size": company_size,
        "lead_value": lead_value,
        "intervention_cost": intervention_cost,
    })


# ============================================================
# DATASET DEFINITIONS
# ============================================================

def synthetic_01(df, rng):

    """
    Easier uplift structure.
    Strong segment signal.
    Useful for:
        - debugging
        - baseline uplift benchmarking
    """

    segment_effect = (
        (df["segment"] == "high_intent") * 1.5
        + (df["segment"] == "mid_intent") * 0.6
        + (df["segment"] == "enterprise") * 1.0
        - (df["segment"] == "bad_fit") * 1.4
    )

    p0_logit = (
        -4
        + 3.5 * df["engagement_score"]
        + 0.30 * df["pages_viewed"]
        + 1.5 * df["form_started"]
        + 2.0 * df["otp_completed"]
        + segment_effect
    )

    p0 = sigmoid(p0_logit)

    tau = (
        0.02
        + 0.14 * (df["segment"] == "mid_intent")
        + 0.10 * df["form_started"]
        - 0.08 * (df["segment"] == "high_intent")
        - 0.12 * (df["segment"] == "bad_fit")
    )

    return p0, tau


def synthetic_02(df, rng):

    """
    Harder dataset.
    Nonlinear interactions.
    Selection bias friendly.
    More realistic overlap issues.
    """

    engagement = df["engagement_score"]

    p0_logit = (
        -5
        + 2.2 * np.sin(engagement * np.pi)
        + 0.45 * np.log1p(df["pages_viewed"])
        + 1.2 * df["otp_completed"]
        + 0.8 * df["form_started"]
        + 1.0 * (df["source"] == "referral")
        - 0.7 * (df["segment"] == "bad_fit")
        + 0.9 * (
            (df["segment"] == "enterprise")
            & (df["company_size"] == "large")
        )
    )

    p0 = sigmoid(p0_logit)

    tau = (
        0.01
        + 0.18 * (
            (engagement > 0.35)
            & (engagement < 0.65)
        )
        - 0.10 * (engagement > 0.85)
        + 0.06 * (
            (df["source"] == "paid")
            & (df["form_started"] == 1)
        )
        - 0.08 * (df["segment"] == "bad_fit")
    )

    return p0, tau


def synthetic_hillstrom(df, rng):

    """
    Hillstrom-inspired structure.

    Treatment:
        email campaign

    Segments:
        men's / women's / no-email-like behavior

    More marketing-oriented uplift shape.
    """

    is_high_engagement = df["engagement_score"] > 0.55

    p0_logit = (
        -4.2
        + 3.0 * df["engagement_score"]
        + 0.25 * df["pages_viewed"]
        + 1.1 * df["form_started"]
        + 1.7 * df["otp_completed"]
        + 0.8 * (df["source"] == "referral")
    )

    p0 = sigmoid(p0_logit)

    tau = (
        0.015
        + 0.10 * (~is_high_engagement)
        + 0.05 * (df["source"] == "paid")
        - 0.06 * is_high_engagement
    )

    return p0, tau


# ============================================================
# TREATMENT ASSIGNMENT
# ============================================================

def assign_treatment(
    df,
    rng,
    treatment_rate=0.5,
    observational=False,
):

    if not observational:

        propensity = np.full(len(df), treatment_rate)

    else:

        propensity = sigmoid(
            -1
            + 2.5 * df["engagement_score"]
            + 1.2 * df["otp_completed"]
            + 0.5 * df["form_started"]
        )

    treatment = sample_binary(propensity, rng)

    return treatment, propensity


# ============================================================
# MAIN GENERATOR
# ============================================================

def generate_synthetic_dataset(
    config: SyntheticConfig
):

    rng = np.random.default_rng(config.random_state)

    df = generate_base_features(
        config.n_samples,
        rng
    )

    # --------------------------------------------------------
    # Dataset selection
    # --------------------------------------------------------

    if config.dataset_type == "synthetic-01":

        p0, tau = synthetic_01(df, rng)

    elif config.dataset_type == "synthetic-02":

        p0, tau = synthetic_02(df, rng)

    elif config.dataset_type == "synthetic-hillstrom":

        p0, tau = synthetic_hillstrom(df, rng)

    else:
        raise ValueError(
            f"Unknown dataset_type: {config.dataset_type}"
        )

    # --------------------------------------------------------
    # Noise
    # --------------------------------------------------------

    if config.noise_std > 0:

        tau = tau + rng.normal(
            0,
            config.noise_std,
            size=len(df)
        )

    # --------------------------------------------------------
    # Clip uplift
    # --------------------------------------------------------

    if config.clip_tau:

        tau = np.clip(tau, -0.30, 0.30)

    p1 = np.clip(
        p0 + tau,
        0,
        1
    )

    # --------------------------------------------------------
    # Treatment assignment
    # --------------------------------------------------------

    treatment, propensity = assign_treatment(
        df=df,
        rng=rng,
        treatment_rate=config.treatment_rate,
        observational=config.observational,
    )

    # --------------------------------------------------------
    # Potential outcomes
    # --------------------------------------------------------

    y0 = sample_binary(p0, rng)

    y1 = sample_binary(p1, rng)

    observed_y = (
        treatment * y1
        + (1 - treatment) * y0
    )

    # --------------------------------------------------------
    # Revenue
    # --------------------------------------------------------

    if config.include_revenue:

        revenue = (
            observed_y
            * df["lead_value"]
            * rng.uniform(0.8, 1.2, size=len(df))
        )

    else:

        revenue = np.zeros(len(df))

    # --------------------------------------------------------
    # Final dataframe
    # --------------------------------------------------------

    df["treatment"] = treatment

    df["propensity"] = propensity

    df["y0_prob"] = p0

    df["tau_true"] = tau

    df["y1_prob"] = p1

    df["y0"] = y0

    df["y1"] = y1

    df["conversion"] = observed_y

    df["revenue"] = revenue

    df["incremental_profit_true"] = (
        tau
        * df["lead_value"]
        - df["intervention_cost"]
    )

    return df


# ============================================================
# EXAMPLE USAGE
# ============================================================

if __name__ == "__main__":

    config = SyntheticConfig(
        dataset_type="synthetic-02",
        n_samples=100_000,
        observational=True,
        random_state=42,
    )

    df = generate_synthetic_dataset(config)

    print(df.head())

    print("\nShape:")
    print(df.shape)

    print("\nConversion Rate:")
    print(df["conversion"].mean())

    print("\nTreatment Rate:")
    print(df["treatment"].mean())

    print("\nAverage True Uplift:")
    print(df["tau_true"].mean())